# Ajuste de hiperparametros do modelo vencedor

Após a comparação entre os três modelos selecionados, a Regressão Logística, com balanceamento da base de treino, foi selecionada como modelo vencedor da etapa anterior. Essa escolha ocorreu porque o modelo apresentou melhor equilíbrio geral entre recall, precision e F1-score da classe 1, que representa internações prolongadas. Apesar do bom desempenho, ainda existe espaço para buscar uma melhora por meio do ajuste de hiperparâmetros.

Nesta etapa, o objetivo é testar diferentes configurações da Regressão Logística para verificar se é possível aumentar o F1-score, melhorar a precision e reduzir falsos positivos, sem comprometer, obviamente, de forma significativa o recall.

Como a base possui grande volume de registros e muitas variáveis após a codificação das categorias, serão testados diferentes valores de regularização e diferentes solvers. O parâmetro `C` será utilizado para controlar a força da regularização do modelo. Valores maiores deixam o modelo mais flexível.

Também será avaliado o uso do solver `saga`, que pode ser mais adequado para bases maiores e permite maior flexibilidade em relação às penalizações. A métrica utilizada para orientar a busca será baseada no equilíbrio entre recall e precision, com atenção especial ao desempenho da classe 1.

Ao final desta etapa, será escolhida a melhor versão do modelo para seguir para a validação temporal com janela expansiva, onde será avaliado se o modelo mantém desempenho estável ao longo dos anos.


In [1]:
from pathlib import Path
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    make_scorer, 
    fbeta_score
)

Vou carregar as bases de treino e teste para trabalhar com o ajuste do modelo.


In [2]:
pasta_saida = Path("dados/processed/splits")

# como vamos usar regressão logistica, não precisamos baixar a base balanceada com SMOTE

x_treino_2020_2022 = pd.read_csv(pasta_saida / "x_treino_2020_2022.csv")
x_teste_2023 = pd.read_csv(pasta_saida / "x_teste_2023.csv")

y_treino_2020_2022 = pd.read_csv(pasta_saida / "y_treino_2020_2022.csv")["internacao_prolongada"]
y_teste_2023 = pd.read_csv(pasta_saida / "y_teste_2023.csv")["internacao_prolongada"]


Como estou usando Regressão Logística, preciso padronizar as variáveis, já que esse modelo é bastante sensível à escala dos dados. Depois disso, vou aplicar o SMOTE para balancear a base de treino.


In [3]:
scaler = StandardScaler()

x_treino_2020_2022_padronizado = x_treino_2020_2022.copy()
x_teste_2023_padronizado = x_teste_2023.copy()

colunas_escalar = ["idade_anos", "ano_internacao", "mes_internacao"]

x_treino_2020_2022_padronizado[colunas_escalar] = scaler.fit_transform(
    x_treino_2020_2022[colunas_escalar]
)

x_teste_2023_padronizado[colunas_escalar] = scaler.transform(
    x_teste_2023[colunas_escalar]
)

In [4]:
smote = SMOTE(random_state=42)

x_treino_2020_2022_padronizado_smote, y_treino_2020_2022_smote = smote.fit_resample(
    x_treino_2020_2022_padronizado,
    y_treino_2020_2022
)

In [ ]:
# Modelo que será utilizado como base nessa fase ao aplicarmos o RandomizedSearchCV

modelo_reglog_base = LogisticRegression(
    random_state=42
)

In [8]:
scoring_reglog = make_scorer(fbeta_score, beta=0.8)

# Regressão Logistica v1

In [9]:
param_grid_reglog_v1 = {
    "C": [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0],
    "penalty": ["l2"],
    "solver": ["lbfgs"],
    "max_iter": [2000, 3000, 5000]
}

In [10]:
modelo_reglog_v1 = RandomizedSearchCV(
    estimator=modelo_reglog_base,
    param_distributions=param_grid_reglog_v1,
    n_iter=12,
    scoring=scoring_reglog,
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=2
)

modelo_reglog_v1.fit(
    x_treino_2020_2022_padronizado_smote,
    y_treino_2020_2022_smote
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END ....C=0.05, max_iter=2000, penalty=l2, solver=lbfgs; total time= 2.5min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END ....C=0.05, max_iter=2000, penalty=l2, solver=lbfgs; total time= 2.6min


...

[CV] END ....C=0.05, max_iter=5000, penalty=l2, solver=lbfgs; total time= 2.6min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END ....C=0.05, max_iter=5000, penalty=l2, solver=lbfgs; total time= 2.4min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'C': [0.05, 0.1, ...], 'max_iter': [2000, 3000, ...], 'penalty': ['l2'], 'solver': ['lbfgs']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",12
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","make_scorer(f...ct', beta=0.8)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the select

In [11]:
modelo_reglog_v1.best_params_

{'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'C': 1.0}

In [12]:
melhor_reglog_v1 = modelo_reglog_v1.best_estimator_

Agora vou testar o modelo.


In [13]:
y_pred_reglog_v1 = modelo_reglog_v1.predict(x_teste_2023_padronizado)

Agora vou avaliar o modelo.


In [14]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2023, y_pred_reglog_v1))

print('\nClassification Report:')
print(classification_report(y_teste_2023, y_pred_reglog_v1))

print('Accuracy:', accuracy_score(y_teste_2023, y_pred_reglog_v1))
print('Recall:', recall_score(y_teste_2023, y_pred_reglog_v1))
print('Precision:', precision_score(y_teste_2023, y_pred_reglog_v1))
print('F1-score:', f1_score(y_teste_2023, y_pred_reglog_v1))

Matriz de confusão:
[[144814  37343]
 [ 14418  32770]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.79      0.85    182157
           1       0.47      0.69      0.56     47188

    accuracy                           0.77    229345
   macro avg       0.69      0.74      0.70    229345
weighted avg       0.82      0.77      0.79    229345

Accuracy: 0.7743094464671129
Recall: 0.6944562176824617
Precision: 0.4673883587922354
F1-score: 0.5587335146332938


O modelo identificou corretamente 33.271 internações prolongadas em 2023, enquanto deixou de identificar 13.916 casos reais da classe 1. O recall foi de aproximadamente 70,51%, indicando que o modelo conseguiu detectar uma parcela relevante das internações prolongadas.

A precision foi de aproximadamente 46,07%, mostrando que, entre os casos classificados como internação prolongada, pouco menos da metade realmente pertencia à classe 1. O F1-score foi de aproximadamente 55,73%, representando um desempenho intermediário entre recall e precision.

De forma geral, o modelo manteve boa capacidade de identificar internações prolongadas, mas ainda apresentou uma quantidade considerável de falsos positivos. Em comparação com a janela temporal anterior, o desempenho apresentou leve queda, o que pode indicar mudança no padrão dos dados ao longo do tempo ou maior dificuldade de generalização para o ano de 2023.


# Regressão Logistica v2

In [18]:
param_grid_reglog_v2 = [
    {
        "solver": ["lbfgs"],
        "penalty": ["l2"],
        "C": [0.7, 1.0, 2.0, 3.0, 5.0],
        "max_iter": [2000, 3000]
    },
    {
        "solver": ["saga"],
        "penalty": ["l2"],
        "C": [0.7, 1.0, 2.0, 3.0, 5.0],
        "max_iter": [3000, 5000],
        "tol": [0.001, 0.0001]
    }
]

In [19]:
modelo_reglog_v2 = RandomizedSearchCV(
    estimator=modelo_reglog_base,
    param_distributions=param_grid_reglog_v2,
    n_iter=12,
    scoring=scoring_reglog,
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=2
)

modelo_reglog_v2.fit(
    x_treino_2020_2022_padronizado_smote,
    y_treino_2020_2022_smote
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END C=5.0, max_iter=3000, penalty=l2, solver=saga, tol=0.0001; total time=15.9min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END C=5.0, max_iter=3000, penalty=l2, solver=saga, tol=0.0001; total time=16.1min


...

[CV] END C=1.0, max_iter=5000, penalty=l2, solver=saga, tol=0.001; total time= 3.4min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END C=1.0, max_iter=5000, penalty=l2, solver=saga, tol=0.001; total time= 3.6min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'C': [0.7, 1.0, ...], 'max_iter': [2000, 3000], 'penalty': ['l2'], 'solver': ['lbfgs']}, {'C': [0.7, 1.0, ...], 'max_iter': [3000, 5000], 'penalty': ['l2'], 'solver': ['saga'], ...}]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",12
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","make_scorer(f...ct', beta=0.8)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum sco

In [20]:
modelo_reglog_v2.best_params_

{'tol': 0.0001, 'solver': 'saga', 'penalty': 'l2', 'max_iter': 3000, 'C': 5.0}

In [22]:
melhor_reglog_v2 = modelo_reglog_v2.best_estimator_

Agora vou testar o modelo.


In [23]:
y_pred_reglog_v2 = modelo_reglog_v2.predict(x_teste_2023_padronizado)

Agora vou avaliar o modelo.


In [24]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2023, y_pred_reglog_v2))

print('\nClassification Report:')
print(classification_report(y_teste_2023, y_pred_reglog_v2))

print('Accuracy:', accuracy_score(y_teste_2023, y_pred_reglog_v2))
print('Recall:', recall_score(y_teste_2023, y_pred_reglog_v2))
print('Precision:', precision_score(y_teste_2023, y_pred_reglog_v2))
print('F1-score:', f1_score(y_teste_2023, y_pred_reglog_v2))

Matriz de confusão:
[[145986  36171]
 [ 14802  32386]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.80      0.85    182157
           1       0.47      0.69      0.56     47188

    accuracy                           0.78    229345
   macro avg       0.69      0.74      0.71    229345
weighted avg       0.82      0.78      0.79    229345

Accuracy: 0.7777453181887549
Recall: 0.6863185555649741
Precision: 0.47239523316364485
F1-score: 0.5596094863709016


O modelo identificou corretamente 32.386 internações prolongadas em 2023, enquanto deixou passar 14.802 casos reais da classe 1. O recall da classe 1 foi de aproximadamente 68,63%, indicando que o modelo conseguiu detectar uma parcela relevante das internações prolongadas.

A precision foi de aproximadamente 47,24%, permitindo observar que, entre os casos classificados como internação prolongada, pouco menos da metade realmente pertencia à classe 1. Isso resultou em uma melhora no F1-score, porém essa melhora veio acompanhada de uma leve queda no recall e de um aumento dos falsos positivos.

De forma geral, o modelo v2 apresentou uma melhora discreta no equilíbrio geral, tornando-se um pouco mais preciso nas previsões positivas. Porém, o ganho foi pequeno, mostrando que os novos ajustes de hiperparâmetros podem ter efeito limitado.


# Regressão Logistica v3

In [27]:
param_grid_reglog_v3 = [
    {
        "solver": ["saga"],
        "penalty": ["l2"],
        "C": [3.0, 5.0, 7.0, 10.0, 15.0],
        "max_iter": [3000, 5000],
        "tol": [0.001, 0.0001]
    },
    {
        "solver": ["saga"],
        "penalty": ["elasticnet"],
        "C": [3.0, 5.0, 7.0, 10.0],
        "l1_ratio": [0.1, 0.3, 0.5],
        "max_iter": [5000],
        "tol": [0.001, 0.0001]
    }
]

In [28]:
modelo_reglog_v3 = RandomizedSearchCV(
    estimator=modelo_reglog_base,
    param_distributions=param_grid_reglog_v3,
    n_iter=12,
    scoring=scoring_reglog,
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=2
)

modelo_reglog_v3.fit(
    x_treino_2020_2022_padronizado_smote,
    y_treino_2020_2022_smote
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END C=7.0, l1_ratio=0.5, max_iter=5000, penalty=elasticnet, solver=saga, tol=0.0001; total time=31.2min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END C=7.0, l1_ratio=0.5, max_iter=5000, penalty=elasticnet, solver=saga, tol=0.0001; total time=37.1min


...

[CV] END C=5.0, l1_ratio=0.1, max_iter=5000, penalty=elasticnet, solver=saga, tol=0.0001; total time=24.4min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[CV] END C=5.0, l1_ratio=0.1, max_iter=5000, penalty=elasticnet, solver=saga, tol=0.0001; total time=25.9min


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'C': [3.0, 5.0, ...], 'max_iter': [3000, 5000], 'penalty': ['l2'], 'solver': ['saga'], ...}, {'C': [3.0, 5.0, ...], 'l1_ratio': [0.1, 0.3, ...], 'max_iter': [5000], 'penalty': ['elasticnet'], ...}]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",12
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","make_scorer(f...ct', beta=0.8)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other t

In [29]:
modelo_reglog_v3.best_params_

{'tol': 0.0001,
 'solver': 'saga',
 'penalty': 'elasticnet',
 'max_iter': 5000,
 'l1_ratio': 0.5,
 'C': 7.0}

In [31]:
melhor_reglog_v3 = modelo_reglog_v3.best_estimator_

Agora vou testar o modelo.


In [32]:
y_pred_reglog_v3 = modelo_reglog_v3.predict(x_teste_2023_padronizado)

Agora vou avaliar o modelo.


In [33]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2023, y_pred_reglog_v3))

print('\nClassification Report:')
print(classification_report(y_teste_2023, y_pred_reglog_v3))

print('Accuracy:', accuracy_score(y_teste_2023, y_pred_reglog_v3))
print('Recall:', recall_score(y_teste_2023, y_pred_reglog_v3))
print('Precision:', precision_score(y_teste_2023, y_pred_reglog_v3))
print('F1-score:', f1_score(y_teste_2023, y_pred_reglog_v3))

Matriz de confusão:
[[146135  36022]
 [ 14868  32320]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.80      0.85    182157
           1       0.47      0.68      0.56     47188

    accuracy                           0.78    229345
   macro avg       0.69      0.74      0.71    229345
weighted avg       0.82      0.78      0.79    229345

Accuracy: 0.7781072183827857
Recall: 0.684919894888531
Precision: 0.4729156302127535
F1-score: 0.5595083528087943


O modelo identificou corretamente 32.320 internações prolongadas em 2023, deixando passar 14.868 casos reais da classe 1. O recall da classe 1 foi de aproximadamente 68,49%, indicando que o modelo conseguiu detectar uma parte relevante das internações prolongadas.

A precision foi de aproximadamente 47,29%, mostrando que, entre os casos classificados como internação prolongada, pouco menos da metade realmente pertencia à classe 1. O modelo ficou com recall e F1-score ligeiramente inferiores aos do modelo anterior.

De forma geral, o modelo v3 apresenta desempenho próximo ao v2, indicando que a escolha da penalização Elastic-Net beneficiou o modelo com uma regularização mista, provavelmente devido ao grande número de variáveis dummy. Porém, os ganhos foram pequenos, mostrando que novos ajustes podem ter retorno limitado.


# Conclusão

Com isso, posso concluir que:

O modelo ajustado da fase anterior apresentou recall de aproximadamente 71,76%, precision de 47,50% e F1-score de 57,16%, valores que utilizei como referência para esta fase.

Assim, testei três novas versões de Regressão Logística usando `RandomizedSearchCV`. A primeira, v1, utilizou penalização `l2` e solver `lbfgs`. A v2 expandiu a busca incluindo o solver `saga`, mais adequado para bases maiores e com muitas variáveis. Já a v3 também testou a penalização `elasticnet`, buscando verificar se uma regularização mista poderia melhorar o desempenho.

Entre as versões, os resultados foram próximos. A v1 teve maior recall, identificando mais internações prolongadas, apesar de gerar mais falsos positivos. A v3 teve maior acurácia e maior precision, além da menor quantidade de falsos positivos, porém com queda no recall. A v2 apresentou o melhor F1-score entre as três versões, indicando o melhor equilíbrio entre recall e precision.

Dessa forma, posso avançar com a Regressão Logística v2, já que ela apresentou melhor equilíbrio entre as três versões. Mesmo assim, no geral, os ganhos entre os modelos foram pequenos, indicando que novos ajustes tendem a trazer melhorias limitadas.

A próxima etapa será uma validação temporal com janela expansiva, com o objetivo de não apenas identificar o desempenho em uma única divisão, mas também analisar se o modelo mantém resultados consistentes ao longo dos anos. Para isso, vou treinar o modelo em períodos acumulados e avaliá-lo no ano seguinte, simulando um cenário mais próximo da aplicação real, em que dados históricos são usados para prever internações futuras.
